# Feature Extraction v3 — Augmentasi train D2 (mean+std)

Membuat **2 salinan ter-augmentasi** untuk tiap file **train D2**, lalu mengekstrak
embedding YAMNet mean+std (2048-d) — konsisten dengan fitur pemenang.

Augmentasi (audio 16 kHz mentah, sebelum YAMNet), RNG di-seed → reproducible:
1. **Time-shift** — `np.roll` acak (± ~0.5 s)
2. **Noise mixing** — campur klip traffic **train** pada SNR acak **5–20 dB**
3. **Peak-normalize** → [-1, 1]
4. **Gain** — kalikan acak **0.6–1.0** (redup, setelah peak-norm agar berefek)

**Anti-leakage:** pool noise hanya dari traffic **split train**. Augmentasi hanya train;
val/test tetap memakai embedding asli (`yamnet_stats_d2.npz`). Embedding train **asli** tidak
dihitung ulang di sini — hanya salinan augmentasi yang di-cache.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ML_DIR = ROOT / "ml"
CACHE_DIR = ML_DIR / "cache"

YAMNET_SR = 16000
DURATION = 3.0
N_SAMPLES = int(YAMNET_SR * DURATION)

SEED = 42
N_AUG = 2                          # salinan augmentasi per file train
SNR_RANGE = (5.0, 20.0)           # dB
SHIFT_MAX = int(0.5 * YAMNET_SR)  # ± 0.5 s
GAIN_RANGE = (0.6, 1.0)

rng = np.random.default_rng(SEED)

train_df = pd.read_csv(ML_DIR / "split_d2_train.csv")
print(f"train D2 : {len(train_df)} file")
print(f"target   : {len(train_df)} x {N_AUG} = {len(train_df) * N_AUG} salinan augmentasi")

train D2 : 1195 file
target   : 1195 x 2 = 2390 salinan augmentasi


## 1 · Muat YAMNet + fungsi audio & augmentasi

In [2]:
t0 = time.time()
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
print(f"YAMNet dimuat ({time.time() - t0:.1f}s)")


def load_raw(path: str) -> np.ndarray:
    """16 kHz mono, panjang tetap N_SAMPLES, TANPA normalisasi (untuk diaugmentasi)."""
    y, _ = librosa.load(path, sr=YAMNET_SR, mono=True)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    return y[:N_SAMPLES].astype(np.float32)


def peak_norm(y: np.ndarray) -> np.ndarray:
    peak = np.abs(y).max()
    return y / peak if peak > 0 else y


def augment(sig: np.ndarray, noise_pool: list, rng) -> np.ndarray:
    """time-shift -> noise mix (SNR acak) -> peak-norm -> gain. Return audio [-1,1]."""
    # 1. time-shift
    y = np.roll(sig, int(rng.integers(-SHIFT_MAX, SHIFT_MAX + 1)))
    # 2. noise mixing pada SNR acak
    noise = noise_pool[int(rng.integers(len(noise_pool)))]
    Ps = np.mean(y ** 2) + 1e-12
    Pn = np.mean(noise ** 2) + 1e-12
    snr = rng.uniform(*SNR_RANGE)
    k = np.sqrt(Ps / (Pn * 10 ** (snr / 10)))
    y = y + k * noise
    # 3. peak-norm
    y = peak_norm(y)
    # 4. gain (redup)
    y = y * rng.uniform(*GAIN_RANGE)
    return y.astype(np.float32), snr


def embed(y: np.ndarray) -> np.ndarray:
    """concat(mean, std) embedding antar-frame -> 2048-d."""
    _s, emb, _sp = yamnet(y)
    e = emb.numpy()
    return np.concatenate([e.mean(axis=0), e.std(axis=0)]).astype(np.float32)

YAMNet dimuat (6.1s)


## 2 · Preload pool noise (traffic train) + sanity

Pool noise = seluruh klip **traffic di split train** (peak-normalized agar level konsisten
sebelum diskalakan ke SNR target).

In [3]:
traffic_train = train_df[train_df.label == "traffic"]
noise_pool = [peak_norm(load_raw(str(ROOT / p))) for p in traffic_train.path]
print(f"pool noise : {len(noise_pool)} klip traffic (train)")

# sanity: augmentasi mengubah sinyal & embedding
demo = load_raw(str(ROOT / train_df.iloc[0].path))
aug, snr = augment(demo, noise_pool, np.random.default_rng(0))
d_orig, d_aug = embed(peak_norm(demo)), embed(aug)
print(f"contoh SNR : {snr:.1f} dB")
print(f"jarak embedding asli vs augmentasi : {np.linalg.norm(d_orig - d_aug):.3f} (harus > 0)")

pool noise : 301 klip traffic (train)


contoh SNR : 9.0 dB
jarak embedding asli vs augmentasi : 3.423 (harus > 0)


## 3 · Ekstrak embedding semua salinan augmentasi

In [4]:
n = len(train_df) * N_AUG
embs = np.zeros((n, 2048), dtype=np.float32)
fnames, labels, sources = [], [], []

print(f"membuat {n} salinan augmentasi ...")
t0 = time.time()
i = 0
for row in train_df.itertuples(index=False):
    sig = load_raw(str(ROOT / row.path))
    for a in range(N_AUG):
        aug, _snr = augment(sig, noise_pool, rng)
        embs[i] = embed(aug)
        fnames.append(f"{Path(row.filename).stem}_aug{a + 1}.wav")
        labels.append(row.label)
        sources.append(row.source_id)
        i += 1
    if i % 500 == 0 or i == n:
        print(f"  {i:>4}/{n}  ({i / (time.time() - t0):.0f} sampel/s)")

out = CACHE_DIR / "yamnet_stats_d2_aug.npz"
np.savez(out, emb=embs, filename=np.array(fnames),
         label=np.array(labels), source_id=np.array(sources))
print(f"\ntersimpan -> {out}  ({embs.shape})  NaN={np.isnan(embs).any()}  {time.time()-t0:.0f}s")

membuat 2390 salinan augmentasi ...


   500/2390  (38 sampel/s)


  1000/2390  (38 sampel/s)


  1500/2390  (39 sampel/s)


  2000/2390  (38 sampel/s)


  2390/2390  (38 sampel/s)

tersimpan -> D:\Coding Vscode\Siren Classification\ml\cache\yamnet_stats_d2_aug.npz  ((2390, 2048))  NaN=False  63s


## 4 · Verifikasi

In [5]:
data = np.load(CACHE_DIR / "yamnet_stats_d2_aug.npz", allow_pickle=True)
print(f"emb        : {data['emb'].shape}")
print(f"NaN        : {np.isnan(data['emb']).any()}")
print(f"per kelas  : {pd.Series(data['label']).value_counts().to_dict()}")
# source augmentasi harus subset dari source train (tidak ada source baru)
train_src = set(pd.read_csv(ML_DIR / 'split_d2_train.csv').source_id)
assert set(data['source_id']).issubset(train_src), "ada source di luar train!"
print("OK — semua source augmentasi berasal dari train (tidak ada kebocoran).")

emb        : (2390, 2048)
NaN        : False
per kelas  : {'police': 648, 'traffic': 602, 'ambulance': 572, 'firetruck': 568}


OK — semua source augmentasi berasal dari train (tidak ada kebocoran).


---

**Selanjutnya:** `10_experiment_augment.ipynb` menggabung train asli + augmentasi ini,
melatih head esf1, dan membandingkan macro-F1 D2 dengan meanstd (0.7983).